# #1 Plot Trees

## Purpose

Plot lineage trees colored by germ layer and cell_type. Also plot example clades and corresponding UMAP embeddings.

## Setup

In [3]:
import treedata as td
import pycea as py
import scanpy as sc
import matplotlib.pyplot as plt 
import pandas as pd
import numpy as np
from devmap.config import set_theme, get_paths, discrete_cmap, germ_layer_palette, stage_palette, celltype_palette
from devmap.utils import save_plot, load_data
from devmap.plots import plot_grouped_characters
import networkx as nx
from copy import deepcopy

set_theme()
base_path, plots_path, results_path = get_paths("trees")

site_names = ["EMX1","HEK3","RNF2"]
edit_palette = {str(i): discrete_cmap[8][i - 1] for i in range(1, 9)}
edit_palette.update({ "!": "#505050","*":"lightgray","-": "white"})


## Load data

In [ ]:
tdata = load_data("topology")

## E9.5-R3 with characters

Select zoom

In [ ]:
zoom_center = 112000
zoom_width = 2000
embryo_tdata = tdata[tdata.obs["embryo"] == "E9.5-R3"].copy()
embryo_tdata.obs["order"] = np.arange(embryo_tdata.shape[0])
embryo_tdata.obs["zoom"] = embryo_tdata.obs["order"].apply(lambda x: abs(x - zoom_center) < zoom_width/2)

Full tree

In [ ]:
fig, ax = plt.subplots(figsize=(3.7,4), dpi = 600)
py.pl.branches(tdata,depth_key ="time", ax = ax, linewidth = .2, color = "clone", legend = False)
plot_grouped_characters(tdata,ax = ax,width = .06,label = True,offset = .02)
py.pl.annotation(tdata,keys = "zoom", legend = False)
ax.tick_params(axis='x', labelsize=6.5)
save_plot(plots_path / "e9.5_characters.svg", fig, rasterize=True)

Zoom

In [ ]:
fig, ax = plt.subplots(figsize=(3.7,2.5), dpi = 600)
py.pl.branches(tdata[tdata.obs.zoom],depth_key ="time", ax = ax, linewidth = .2, color = "clone", legend = False)
plot_grouped_characters(tdata[tdata.obs.zoom],ax = ax,width = .06,label = True,offset = .02)
ax.tick_params(axis='x', labelsize=6.5)
save_plot(plots_path / "e9.5_characters_zoom.svg", fig, rasterize=True)

## E9.5-R3 clade

In [ ]:
fig, ax = plt.subplots(figsize=(1.8,1.8),dpi=1200)
i = 60201
py.pl.branches(embryo_tdata[i:i + 5000], depth_key="time", ax = ax,linewidth=0.2, color = stage_palette["E9.5"])
py.pl.annotation(embryo_tdata, ax = ax, keys = ["germ_layer"], palette=germ_layer_palette, legend = False, gap = .02, width = .15, label = False)
py.pl.annotation(embryo_tdata, ax = ax, keys = ["cell_type"], palette=celltype_palette, gap = .02, width = .15, label = False)
save_plot(plots_path / "e9.5-r3_zoom1.svg", fig, rasterize=True, dpi=2000)

In [42]:
fig, ax = plt.subplots(figsize=(4.5, 4.5), dpi=600)
sc.pl.umap(tdata, palette=["lightgray"], legend_loc = None, s = .2, ax = ax, show = False, frameon=False, title = "")
sc.pl.umap(embryo_tdata[i:i + 5000], color = ["cell_type"], legend_loc = None, s = 10, 
           ax = ax,frameon=False, title = "", palette=celltype_palette)
save_plot(plots_path / "e9.5-r3_zoom1_umap.svg", fig, rasterize=True)

## E9.5-R3 subclade

In [205]:
fig, ax = plt.subplots(figsize=(1.8,1.8),dpi=1200)
py.pl.branches(embryo_tdata[i+2250:i + 2750], depth_key="time", ax = ax,linewidth=0.2, color = stage_palette["E9.5"])
py.pl.annotation(embryo_tdata, ax = ax, keys = ["germ_layer"], palette=germ_layer_palette, legend = False, gap = .02, width = .15, label = False)
py.pl.annotation(embryo_tdata, ax = ax, keys = ["cell_type"], palette=celltype_palette, gap = .02, width = .15, label = False)
save_plot(plots_path / "e9.5-r3_zoom2.svg", fig, rasterize=True, dpi=2000)

In [41]:
fig, ax = plt.subplots(figsize=(4.5, 4.5), dpi=600)
sc.pl.umap(tdata, palette=["lightgray"], legend_loc = None, s = .2, ax = ax, show = False, frameon=False, title = "")
sc.pl.umap(embryo_tdata[i+2250:i + 2750], color = ["cell_type"], legend_loc = None, s = 20, 
           ax = ax,frameon=False, title = "", palette=celltype_palette)
save_plot(plots_path / "e9.5-r3_zoom2_umap.svg", fig, rasterize=True)

## Trees for all embryos

In [ ]:
for embryo in tdata.obs["embryo"].unique():
    embryo_tdata = tdata[tdata.obs["embryo"] == embryo].copy()
    n_cells = len(embryo_tdata.obs)
    scale = np.sqrt(n_cells/40000)+.5
    print(f"Figsize for {embryo} with {n_cells} cells: {scale}")
    fig, ax = plt.subplots(figsize=(scale,scale), dpi=2000, subplot_kw={"projection": "polar"})
    stage = embryo.split("-")[0]
    width = 1/(scale*3)
    py.pl.branches(embryo_tdata,polar = True, depth_key="time", ax = ax, linewidth=.2, color = stage_palette[stage])
    py.pl.annotation(embryo_tdata,keys = ["germ_layer"], width = width, gap = width / 5, palette=germ_layer_palette, legend = False)
    py.pl.annotation(embryo_tdata,keys = ["cell_type"], width = width, gap = width / 5, palette=celltype_palette, legend = False)
    save_plot(plots_path / f"{embryo}_tree.svg", fig, rasterize=True, dpi=2000)